In [ ]:
import numpy as np
from sklearn.feature_selection import f_classif

# Compute F-values and p-values
F_values, p_values = f_classif(XX, y1)

# Create a DataFrame with OTUs, F-values, and Genus
feature_scores = pd.DataFrame({
    'OTU': XX.columns,
    'F_value': F_values,
    'p_value': p_values
})
feature_scores['Genus'] = feature_scores['OTU'].map(otu_to_genus)

# Group by Genus and select the OTU with the highest F-value in each genus
top_otus_per_genus = feature_scores.sort_values('F_value', ascending=False).groupby('Genus').first().reset_index()

# Select the top N genera
N = 50
selected_otus = top_otus_per_genus.head(N)['OTU']

# Create a reduced OTU table
reduced_otu_table = shared[selected_otus]

In [12]:
import pandas as pd

# Assume 'shared' is your OTU abundance DataFrame and 'meta' is your metadata DataFrame
shared = pd.read_table("../data/baxter.0.03.subsample.shared")
meta = pd.read_table("../data/metadata.tsv")

# Step 1: Map OTUs to Genus Names
taxonomy_df = pd.read_table('../data/baxter.taxonomy', sep='\t')

def extract_genus(taxonomy_str):
    taxa_levels = taxonomy_str.strip(';').split(';')
    genus_info = taxa_levels[-1]
    genus_name = genus_info.split('(')[0]
    return genus_name

taxonomy_df['Genus'] = taxonomy_df['Taxonomy'].apply(extract_genus)
otu_to_genus = dict(zip(taxonomy_df['OTU'], taxonomy_df['Genus']))

# Step 2: Replace OTU IDs with Genus Names
otu_columns = [col for col in shared.columns if col.startswith('Otu')]
shared_genus = shared[otu_columns].copy()
shared_genus.columns = [otu_to_genus.get(otu, 'Unknown') for otu in otu_columns]

# Step 3: Aggregate Abundances by Genus
shared_genus_aggregated = shared_genus.groupby(shared_genus.columns, axis=1).sum()

# Step 4: Merge with Metadata
data = pd.merge(meta, shared_genus_aggregated, left_on='sample', right_index=True)

# Step 5: Prepare Features and Labels
additional_columns = ['Age', 'Gender', 'Smoke', 'BMI']
X = data[additional_columns + list(shared_genus_aggregated.columns)]
y = data['Diabetic']

# Step 6: Proceed with Machine Learning
# Example: Using Random Forest Classifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

/var/folders/gv/py_qrkp162g_1kqt0zrs11qc0000gn/T/ipykernel_15546/1320070816.py:25: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  shared_genus_aggregated = shared_genus.groupby(shared_genus.columns, axis=1).sum()


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate model
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


In [3]:
X

,Age,Gender,Smoke,BMI,Abiotrophia,Acidaminococcaceae_unclassified,Acidaminococcus,Acidobacteria_Gp4_unclassified,Acidobacterium_unclassified,Acidovorax,...,Turicibacter,Vagococcus,Varibaculum,Veillonella,Veillonellaceae_unclassified,Verrucomicrobium,Victivallis,Weissella,Wolinella,Yersinia


In [ ]:

# Feature Importance
import matplotlib.pyplot as plt

importances = model.feature_importances_
feature_names = X.columns

# Create a DataFrame for visualization
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sort features by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Plot top 20 features
top_features = feature_importance_df.head(20)

plt.figure(figsize=(10, 8))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Top 20 Features')
plt.gca().invert_yaxis()
plt.show()

In [4]:
import pandas as pd

# Read the taxonomy data
taxonomy_df = pd.read_table("../data/baxter.taxonomy", sep='\t')

# Function to extract genus from taxonomy string
def extract_genus(taxonomy_str):
    taxa_levels = taxonomy_str.strip(';').split(';')
    genus_info = taxa_levels[-1]  # Get the last item (genus)
    genus_name = genus_info.split('(')[0]  # Remove confidence score
    return genus_name

# Apply the function to extract genus names
taxonomy_df['Genus'] = taxonomy_df['Taxonomy'].apply(extract_genus)

# Create a mapping from OTU IDs to Genus names
otu_to_genus = dict(zip(taxonomy_df['OTU'], taxonomy_df['Genus']))

In [5]:
# Create new feature names by combining OTU IDs with Genus names
otu_columns = [col for col in shared.columns if col.startswith('Otu')]

# Create a mapping of OTU IDs to composite names
otu_to_composite_name = {
    otu: f"{otu}_{otu_to_genus.get(otu, 'Unknown')}" for otu in otu_columns
}

# Rename OTU columns in the DataFrame
shared_composite = shared[otu_columns].copy()
shared_composite.rename(columns=otu_to_composite_name, inplace=True)

In [6]:
# Merge with metadata
data = pd.merge(meta, shared_composite, left_on='sample', right_index=True)

In [7]:
# Define features (X) and labels (y)
additional_columns = ['Age', 'Gender', 'Smoke', 'BMI']
X = data.drop(['sample', 'Diabetic'], axis=1)
X = X[additional_columns + list(shared_composite.columns)]
y = data['Diabetic']

In [14]:
shared_composite

,Otu00001(Blautia),Otu00002(Bacteroides),Otu00003(Bacteroides),Otu00004(Akkermansia),Otu00005(Bacteroides),Otu00006(Roseburia),Otu00007(Blautia),Otu00008(Ruminococcus),Otu00009(Clostridiales_unclassified),Otu00010(Anaerostipes),...,Otu11258(Bacteria_unclassified),Otu11263(Bacteria_unclassified),Otu11269(Bacteria_unclassified),Otu11270(Bacteria_unclassified),Otu11271(Bacteria_unclassified),Otu11274(Bacteria_unclassified),Otu11277(Bacteria_unclassified),Otu11278(Bacteria_unclassified),Otu11280(Bacteria_unclassified),Otu11281(Bacteria_unclassified)
0,350,268,213,1,208,230,70,230,235,64,...,0,0,0,0,0,0,0,0,0,0
1,568,1320,13,293,671,103,48,204,119,115,...,0,0,0,0,0,1,0,0,0,0
2,151,756,802,556,145,271,57,176,37,710,...,0,0,0,0,0,0,0,0,0,0
3,299,30,1018,0,25,99,75,78,255,197,...,0,0,0,0,0,0,0,0,0,0
4,1409,174,0,3,2,1136,296,1,537,533,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,393,451,111,1538,33,1126,423,3,382,114,...,0,0,0,0,0,0,0,0,0,0
486,1755,23,1,1280,473,2182,0,0,221,234,...,0,0,0,0,0,0,0,0,0,0
487,1005,123,131,22,302,164,399,678,130,175,...,0,0,0,0,0,0,0,0,0,0
488,25,220,384,228,37,4,19,0,72,23,...,0,0,0,0,0,0,0,0,0,0


In [9]:
import pandas as pd
import numpy as np

# 1. Modify Composite Names
# Read the taxonomy data
taxonomy_df = pd.read_table("../data/baxter.taxonomy", sep='\t')

# Function to extract genus from taxonomy string
def extract_genus(taxonomy_str):
    taxa_levels = taxonomy_str.strip(';').split(';')
    genus_info = taxa_levels[-1]  # Get the last item (genus)
    genus_name = genus_info.split('(')[0]  # Remove confidence score
    return genus_name

# Apply the function to extract genus names
taxonomy_df['Genus'] = taxonomy_df['Taxonomy'].apply(extract_genus)

# Create a mapping from OTU IDs to Genus names
otu_to_genus = dict(zip(taxonomy_df['OTU'], taxonomy_df['Genus']))

# Create new feature names with desired nomenclature
otu_columns = [col for col in shared.columns if col.startswith('Otu')]

# Create a mapping of OTU IDs to composite names
otu_to_composite_name = {
    otu: f"{otu}({otu_to_genus.get(otu, 'Unknown')})" for otu in otu_columns
}

# Rename OTU columns in the DataFrame
shared_composite = shared[otu_columns].copy()
shared_composite.rename(columns=otu_to_composite_name, inplace=True)

# 2. Feature Selection Using Random Forest
# Merge with metadata
data = pd.merge(meta, shared_composite, left_on='sample', right_index=True)

# Define features (X) and labels (y)
X_otu = data.drop(['sample', 'Diabetic'], axis=1)
y = data['Diabetic']

# Handle missing values if any
X_otu.fillna(0, inplace=True)

In [15]:
data

,sample,fit_result,Site,Dx_Bin,dx,Hx_Prev,Hx_of_Polyps,Age,Gender,Smoke,...,Turicibacter,Vagococcus,Varibaculum,Veillonella,Veillonellaceae_unclassified,Verrucomicrobium,Victivallis,Weissella,Wolinella,Yersinia


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train_otu, X_test_otu, y_train, y_test = train_test_split(
    X_otu, y, test_size=0.2, random_state=42
)

# Train a Random Forest classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_otu, y_train)

# Get feature importances
importances = rf.feature_importances_
feature_importances = pd.DataFrame({
    'Feature': X_otu.columns,
    'Importance': importances
})

# Sort features by importance
feature_importances.sort_values(by='Importance', ascending=False, inplace=True)

# Select top N features
top_n = 50
top_features = feature_importances['Feature'].head(top_n).tolist()

# Create a reduced feature set with top features
X_top = X_otu[top_features]

# 3. Merge with Additional Columns
additional_columns = ['Age', 'Gender', 'Smoke', 'BMI']
X_additional = data[additional_columns]

# Combine top OTU features with additional columns
X_final = pd.concat([X_top.reset_index(drop=True), X_additional.reset_index(drop=True)], axis=1)
y_final = y.reset_index(drop=True)

In [ ]:
# 4. Correlation Plot
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = X_final.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False)
plt.title('Correlation Matrix of Features')
plt.show()

# 5. Develop Machine Learning Models
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': lgb.LGBMClassifier(random_state=42)
}

# Train and evaluate models
model_metrics = {}

for name, model in models.items():
    print(f"Training {name}...")
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    model_metrics[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC AUC': roc_auc
    }

# Display metrics
metrics_df = pd.DataFrame(model_metrics).T
print(metrics_df)

# 6. Plot ROC Curves
plt.figure(figsize=(10, 8))

for name, model in models.items():
    if name == 'Logistic Regression':
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {model_metrics[name]['ROC AUC']:.2f})")

plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Diabetes Prediction Models')
plt.legend(loc='lower right')
plt.show()

In [ ]:

# 7. SHAP Analysis
import shap

# SHAP analysis for Random Forest
explainer_rf = shap.TreeExplainer(models['Random Forest'])
shap_values_rf = explainer_rf.shap_values(X_test)

# Plot SHAP summary plot
shap.summary_plot(shap_values_rf[1], X_test, plot_type='bar', show=False)
plt.title('SHAP Summary Plot for Random Forest')
plt.show()

# SHAP analysis for XGBoost
explainer_xgb = shap.TreeExplainer(models['XGBoost'])
shap_values_xgb = explainer_xgb.shap_values(X_test)

# Plot SHAP summary plot
shap.summary_plot(shap_values_xgb, X_test, plot_type='bar', show=False)
plt.title('SHAP Summary Plot for XGBoost')
plt.show()